<a href="https://colab.research.google.com/github/ancestor9/mathematics-for-machine-learning/blob/main/pytorch_recap/Task_01_MNIST_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. MNIST 데이터셋, 딥러닝의 'Hello, World!' 👋

MNIST(Modified National Institute of Standards and Technology) 데이터셋은 0부터 9까지의 숫자가 손글씨로 쓰인 흑백 이미지 데이터셋입니다.

각 이미지는 28x28 픽셀 크기이며, 총 60,000개의 학습 데이터와 10,000개의 테스트 데이터로 구성되어 있습니다.



### 2. PyTorch로 MNIST 분류하기

이번 실습에서는 다층 퍼셉트론(MLP) 모델을 사용하여 MNIST 데이터셋을 분류해 보겠습니다. 선형 회귀와 마찬가지로 다음의 5단계를 따라 진행합니다.


1. 데이터 준비: MNIST 데이터셋을 불러오고 전처리합니다.

2. 모델 설계: 입력, 은닉, 출력층으로 구성된 MLP 모델을 정의합니다.

3. 손실 함수와 옵티마이저 정의: 분류 문제에 맞는 손실 함수와 옵티마이저를 설정합니다.

4. 모델 학습: 준비된 데이터로 모델을 학습시킵니다.

5. 모델 평가: 학습된 모델의 성능(정확도)을 평가합니다.



Step 1: 데이터 준비

PyTorch는 torchvision 라이브러리를 통해 MNIST 데이터셋을 매우 쉽게 불러올 수 있습니다. 데이터셋을 불러올 때, 텐서로 변환하고 정규화(Normalization)하는 변환(Transform) 과정을 함께 적용합니다.



In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# 텐서로 변환하고 -1~1 범위로 정규화하는 변환기 정의
transform = transforms.Compose([
    transforms.ToTensor(), # 이미지를 텐서로 변환
    transforms.Normalize((0.5,), (0.5,)) # (평균, 표준편차)로 정규화
])

# 학습 데이터셋 및 데이터로더 불러오기
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# 테스트 데이터셋 및 데이터로더 불러오기
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 43.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.11MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.94MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.89MB/s]


Step 2: 모델 설계

28x28 픽셀 이미지를 MLP의 입력으로 사용하기 위해, 이미지를 1차원 벡터로 펼쳐야 합니다. 28 times 28=784이므로 입력층의 뉴런 수는 784개가 됩니다. 출력층은 0부터 9까지 10개의 클래스를 분류해야 하므로 10개의 뉴런을 가집니다.



In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 입력층: 784 (28*28)
        self.fc1 = nn.Linear(784, 128)
        # 은닉층
        self.fc2 = nn.Linear(128, 64)
        # 출력층: 10 (0~9)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        # 이미지를 1차원 벡터로 펼치기 (flatten)
        x = x.view(-1, 28 * 28)
        # 순전파 과정
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x) # 출력층에는 활성화 함수를 사용하지 않습니다.
        return x


Step 3 & 4: 손실 함수, 옵티마이저 정의 및 학습

분류 문제에서는 주로 CrossEntropyLoss를 사용합니다. 옵티마이저는 Adam을 사용하겠습니다.



In [ ]:
# 모델, 손실 함수, 옵티마이저 정의
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습 루프
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(train_loader):
        inputs, labels = data

        # 순전파
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # 역전파
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'에포크 [{epoch+1}/{num_epochs}], 손실: {running_loss/len(train_loader):.4f}')
print("학습 완료!")


에포크 [1/5], 손실: 0.3824
에포크 [2/5], 손실: 0.1823
에포크 [3/5], 손실: 0.1351
에포크 [4/5], 손실: 0.1080
에포크 [5/5], 손실: 0.0958
학습 완료!


Step 5: 모델 평가

학습된 모델의 성능을 테스트 데이터셋으로 평가합니다. 정확도(Accuracy)를 계산하여 모델이 얼마나 잘 분류하는지 확인합니다.



In [ ]:
correct = 0
total = 0
with torch.no_grad(): # 평가 단계에서는 기울기 계산이 필요 없으므로 비활성화
    for data in test_loader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'\n테스트 데이터셋에서 모델의 정확도: {100 * correct / total:.2f}%')


테스트 데이터셋에서 모델의 정확도: 96.82%


## 실습 문제
1. 위 코드에서 batch_size를 64에서 256으로 변경했을 때, 학습 속도와 최종 정확도에 어떤 변화가 있을지 예측하고 코드를 실행하여 확인해 보세요.

2. MLP 클래스의 은닉층 수를 하나 더 추가하거나, 은닉층의 뉴런 수를 변경해 보세요. 모델의 복잡도가 정확도에 어떤 영향을 미치는지 실험해 봅시다.

3. optimizer를 torch.optim.SGD로 변경했을 때, 학습률(lr)을 어떻게 조절해야 좋은 성능을 낼 수 있을지 실험해 보세요.